# MBTA Bus Delay Modeling & Evaluation
Predicting route-day mean delay (minutes) across MBTA bus routes in 2024 using rolling historical lags, ridership, scheduled travel time, and calendar/holiday indicators.

In [6]:
import warnings, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score, roc_auc_score, roc_curve,
    f1_score, precision_score, recall_score, accuracy_score, confusion_matrix
)
warnings.filterwarnings("ignore")

SEED = 506

# Safer path resolution with a clear error message if data is missing
candidate_paths = [Path("data/processed"), Path("."), Path("/mnt/user-data/uploads")]
DATA = next((p for p in candidate_paths if (p / "route_day_delay.csv").exists()), None)
if DATA is None:
    raise FileNotFoundError(
        "Could not find 'route_day_delay.csv' in 'data/processed/', current directory, or '/mnt/user-data/uploads/'."
    )

FIGS = Path("figures")
FIGS.mkdir(exist_ok=True)

TARGET = {"14", "15", "17", "22", "23", "24", "26", "28", "29", "31", "33", "42", "44", "45", "111"}
ROUTE_ALIAS = {
    "SL1": "741", "SL2": "742", "SL3": "743", "SL4": "751",
    "SL5": "749", "SLW": "746", "CT1": "701", "CT2": "747", "CT3": "708"
}

BLUE, ORANGE, GRAY, INK, INK2, MUTED, GRID, BASELINE = (
    "#2a78d6", "#eb6834", "#a9a79f", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
)
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "font.family": "sans-serif", "font.sans-serif": ["Segoe UI", "DejaVu Sans"],
    "font.size": 10.5, "text.color": INK, "axes.labelcolor": INK2,
    "xtick.color": INK2, "ytick.color": INK2, "axes.edgecolor": BASELINE,
    "axes.linewidth": .8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": .7, "axes.axisbelow": True,
    "xtick.major.size": 0, "ytick.major.size": 0, "legend.frameon": False,
    "axes.titlelocation": "left", "axes.titlesize": 12, "axes.titleweight": "semibold"
})

def source(fig, text):
    fig.text(0.01, -0.015, text, color=MUTED, fontsize=8.5, va="top")

def save(fig, name):
    fig.savefig(FIGS / name)
    plt.show()

def norm_route(s):
    s = str(s).strip().strip('"').upper()
    return ROUTE_ALIAS.get(s) or s.lstrip("0") or s

print("data folder:", DATA)

FileNotFoundError: Could not find 'route_day_delay.csv' in 'data/processed/', current directory, or '/mnt/user-data/uploads/'.

In [ ]:
raw = pd.read_csv(DATA / "route_day_delay.csv", parse_dates=["date"], dtype={"route": str})
raw["route"] = raw.route.map(norm_route)
raw["delay"] = raw.sum_delay / raw.n / 60

for label, g in [
    ("Jan-Feb 2020", raw[raw.date.dt.year == 2020]),
    ("Jan-Feb 2024", raw[(raw.date.dt.year == 2024) & (raw.date.dt.month <= 2)]),
    ("Jan-Nov 2024", raw[raw.date.dt.year == 2024])
]:
    print(f"{label}: delay {g.sum_delay.sum() / g.n.sum() / 60:.2f} min, on time {100 * g.on_time.sum() / g.n.sum():.1f}%")

def build_features(raw):
    d = raw[raw.date.dt.year == 2024].copy()

    # Vectorized 2020 baseline delay (avoids pandas groupby.apply issues)
    g20 = raw[raw.date.dt.year == 2020].groupby("route")[["sum_delay", "n"]].sum()
    d20 = (g20["sum_delay"] / g20["n"] / 60).rename("delay_2020")

    rs = pd.read_csv(DATA / "ridership_route_season.csv", dtype={"route": str})
    rs["route"] = rs.route.map(norm_route)
    board = rs[rs.season == "Fall 2023"].groupby("route").boardings.sum().rename("boardings_f23")

    tt = pd.read_csv(DATA / "travel_time_all_routes.csv", dtype={"route": str})
    tt["route"] = tt.route.map(norm_route)
    sched = tt.set_index("route")["scheduled (min)"].rename("sched_trip_min")

    d = d.merge(d20, on="route", how="left").merge(board, on="route", how="left").merge(sched, on="route", how="left")
    d["log_board"] = np.log1p(d.boardings_f23)
    d["is_target"] = d.route.isin(TARGET).astype(int)

    # Calendar
    d["dow"] = d.date.dt.dayofweek
    for k in range(5):
        d[f"dow{k}"] = (d.dow == k).astype(int)

    holidays = pd.to_datetime([
        "2024-01-01", "2024-01-15", "2024-02-19", "2024-04-15", "2024-05-27",
        "2024-06-19", "2024-07-04", "2024-09-02", "2024-10-14", "2024-11-11",
        "2024-11-28", "2024-11-29"
    ])
    d["holiday"] = d.date.isin(holidays).astype(int)
    d["near_holiday"] = d.date.map(lambda x: int(any(abs((x - h).days) == 1 for h in holidays)))
    d["school_vacation"] = 0
    for a, b in [("2024-02-19", "2024-02-23"), ("2024-04-15", "2024-04-19")]:
        d.loc[(d.date >= a) & (d.date <= b), "school_vacation"] = 1
    d["school_out"] = ((d.date >= "2024-06-26") & (d.date <= "2024-09-03")).astype(int)

    # Lagged delay (closed="left" prevents target leakage)
    parts = []
    for _, g in d.sort_values("date").groupby("route"):
        g = g.set_index("date")
        s, c = g.sum_delay, g.n
        g["lag28"] = s.rolling("28D", closed="left").sum() / c.rolling("28D", closed="left").sum() / 60
        g["lag7"] = s.rolling("7D", closed="left").sum() / c.rolling("7D", closed="left").sum() / 60
        g["hist_days"] = c.rolling("28D", closed="left").count()
        g["log_nlag"] = np.log1p(c.rolling("28D", closed="left").sum() / g.hist_days)
        g["lag_sameweekday"] = g.delay.shift(1, freq="7D").reindex(g.index)
        parts.append(g.reset_index())

    d = pd.concat(parts, ignore_index=True)
    d = d[d.hist_days >= 5].copy()
    d["lag_sameweekday"] = d.lag_sameweekday.fillna(d.lag28)
    return d.sort_values(["date", "route"]).reset_index(drop=True)

d = build_features(raw)
FEATURES = [
    "lag7", "lag28", "lag_sameweekday", "log_nlag", "delay_2020", "log_board",
    "sched_trip_min", "is_target", "holiday", "near_holiday", "school_vacation",
    "school_out", "dow0", "dow1", "dow2", "dow3", "dow4"
]
print(f"{len(d):,} route-days, {d.route.nunique()} routes, {d.date.min().date()} to {d.date.max().date()}")
print("missing feature values (filled with the training median inside each model):", d[FEATURES].isna().sum()[lambda s: s > 0].to_dict())
print(f"target: mean {d.delay.mean():.2f} min, sd {d.delay.std():.2f}, 5th-95th percentile {d.delay.quantile(.05):.1f} to {d.delay.quantile(.95):.1f}")

NameError: name 'pd' is not defined

In [ ]:
# Train / Holdout Split (Oct 1, 2024 cutoff)
CUTOFF = pd.Timestamp("2024-10-01")
train = d[d.date < CUTOFF].copy()
test = d[d.date >= CUTOFF].copy()

X_tr, y_tr = train[FEATURES], train["delay"]
X_te, y_te = test[FEATURES], test["delay"]
print(f"Train: {len(train):,} rows ({train.date.min().date()} to {train.date.max().date()})")
print(f"Test:  {len(test):,} rows ({test.date.min().date()} to {test.date.max().date()})")

MODELS = {
    "Ridge regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=10.0)),
    "KNN (k=25)": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), KNeighborsRegressor(n_neighbors=25, weights="distance")),
    "Decision tree": make_pipeline(SimpleImputer(strategy="median"), DecisionTreeRegressor(max_depth=6, min_samples_leaf=25, random_state=SEED)),
    "Random forest": make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, min_samples_leaf=10, max_features=0.33, n_jobs=-1, random_state=SEED)),
    "Gradient boosting": make_pipeline(SimpleImputer(strategy="median"), GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=SEED)),
}

# Expanding-window time-series CV on training set
cv_cutoffs = [pd.Timestamp("2024-06-01"), pd.Timestamp("2024-07-01"), pd.Timestamp("2024-08-01"), pd.Timestamp("2024-09-01")]
cv_scores = {name: [] for name in MODELS}

for i in range(len(cv_cutoffs)):
    c_start = cv_cutoffs[i]
    c_end = cv_cutoffs[i + 1] if i + 1 < len(cv_cutoffs) else CUTOFF
    tr_idx = train.date < c_start
    va_idx = (train.date >= c_start) & (train.date < c_end)
    for name, model in MODELS.items():
        model.fit(train.loc[tr_idx, FEATURES], train.loc[tr_idx, "delay"])
        p = model.predict(train.loc[va_idx, FEATURES])
        cv_scores[name].append(mean_absolute_error(train.loc[va_idx, "delay"], p))

cv_means = {k: np.mean(v) for k, v in cv_scores.items()}
SELECTED = min(cv_means, key=cv_means.get)
print("\nExpanding-window CV MAE:")
for k, v in sorted(cv_means.items(), key=lambda x: x[1]):
    print(f"  {k:<22} {v:.3f} min")
print("Selected model:", SELECTED)

NameError: name 'pd' is not defined

In [ ]:
# Fit all models on full training set and evaluate on Oct-Nov 2024 holdout
pred = {
    "Global mean baseline": np.full(len(test), y_tr.mean()),
    "Last 28 days rule": test["lag28"].fillna(y_tr.mean()).values,
    "Last 7 days rule": test["lag7"].fillna(test["lag28"]).fillna(y_tr.mean()).values,
}

rows = []
for name, p in pred.items():
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_te, p),
        "RMSE": np.sqrt(mean_squared_error(y_te, p)),
        "R2": r2_score(y_te, p)
    })

for name, model in MODELS.items():
    model.fit(X_tr, y_tr)
    p = model.predict(X_te)
    pred[name] = p
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_te, p),
        "RMSE": np.sqrt(mean_squared_error(y_te, p)),
        "R2": r2_score(y_te, p)
    })

res = pd.DataFrame(rows).set_index("Model").sort_values("MAE")
print(res.round(3).to_string())

In [ ]:
# Figure 11. Model comparison
fig, ax = plt.subplots(figsize=(7.5, 4.2))
order = res.sort_values("MAE", ascending=False)
colors = [BLUE if k in MODELS else GRAY for k in order.index]
colors[list(order.index).index(SELECTED)] = ORANGE

bars = ax.barh(order.index, order["MAE"], color=colors, height=0.62)
ax.set_xlabel("Holdout Mean Absolute Error (minutes, Oct 1 – Nov 29, 2024)")
ax.set_title("Figure 11. Fitted models beat the 7-day rolling rule by 0.1–0.2 minutes")
ax.set_xlim(0, order["MAE"].max() * 1.18)

for bar, (_, row) in zip(bars, order.iterrows()):
    ax.text(
        bar.get_width() + 0.03,
        bar.get_y() + bar.get_height() / 2,
        f"{row['MAE']:.2f} min (R²={row['R2']:.2f})",
        va="center", fontsize=9.5, color=INK
    )

source(fig, "Source: MBTA 2024 route-day delay data. Orange = CV-selected model.")
save(fig, "fig11_model_comparison.png")

In [ ]:
# Error breakdown (Holidays vs Regular Days & Target vs Non-Target Routes)
test["pred"] = pred[SELECTED]
test["abs_err"] = (test["delay"] - test["pred"]).abs()

print("Holdout error by day type:")
print(test.groupby("holiday")[["delay", "pred", "abs_err"]].agg(
    days=("delay", "count"),
    actual_mean=("delay", "mean"),
    pred_mean=("pred", "mean"),
    MAE=("abs_err", "mean")
).round(2).to_string())

print("\nHoldout error by equity target route status:")
print(test.groupby("is_target")[["delay", "pred", "abs_err"]].agg(
    route_days=("delay", "count"),
    actual_mean=("delay", "mean"),
    pred_mean=("pred", "mean"),
    MAE=("abs_err", "mean")
).round(2).to_string())

In [ ]:
# High-delay classifier (>= 5 min mean delay)
y_tr_cls = (train["delay"] >= 5.0).astype(int)
y_te_cls = (test["delay"] >= 5.0).astype(int)

clf = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(n_estimators=200, min_samples_leaf=10, max_features=0.33, n_jobs=-1, random_state=SEED)
)
clf.fit(train[FEATURES], y_tr_cls)
prob_rf = clf.predict_proba(test[FEATURES])[:, 1]
prob_lag7 = test["lag7"].fillna(test["lag28"]).fillna(train["delay"].mean())

print(f"High-delay (>=5 min) prevalence in holdout: {y_te_cls.mean():.1%}")
print(f"Random Forest Classifier AUC: {roc_auc_score(y_te_cls, prob_rf):.3f}")
print(f"Last 7 days baseline AUC:     {roc_auc_score(y_te_cls, prob_lag7):.3f}")

NameError: name 'train' is not defined